# Dense Retrieval Basics

This notebook explores the fundamentals of dense retrieval, including text embeddings, cosine similarity, Top-K ranking, and vectorized search.

## 1. Document Embedding Generation

Load a pretrained Sentence Transformer model and encode a small collection of documents into dense vector representations.

Each document is represented by a 384-dimensional embedding, which will be used for similarity-based retrieval.

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

documents = [
    "A dog is running in the park.",
    "A puppy is playing outside.",
    "The database stores information.",
    "Search engines retrieve relevant documents.",
    "A cat is sleeping on the sofa."
]

document_embeddings = model.encode(documents)

print("Embedding shape:", document_embeddings.shape)
print("First 10 values:", document_embeddings[0][:10])

C:\Users\weiting\anaconda3\envs\dense-retrieval\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\weiting\anaconda3\envs\dense-retrieval\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\weiting\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as 

Embedding shape: (5, 384)
First 10 values: [ 0.04797399 -0.05242923  0.04978537 -0.00862706  0.06776327 -0.03118443
 -0.00056124  0.01672406  0.05382127  0.03046024]


## 2. Query Embedding and Cosine Similarity

Dense retrieval measures the semantic similarity between a query and document embeddings.

We use **cosine similarity** to measure the similarity between two vectors:

$$
\text{cosine}(q,d)=\frac{q\cdot d}{\|q\|\|d\|}
$$

where:
- $q$ is the query embedding.
- $d$ is a document embedding.
- $q\cdot d$ is their dot product.
- $\|q\|$ and $\|d\|$ are their L2 norms.

In this section, we manually implement cosine similarity and calculate the similarity score between the query and each document.

In [2]:
import numpy as np

query = "How do search engines find information?"

query_embedding = model.encode(query)

def cosine_similarity(a, b):
    dot_product = np.dot(a, b)
    norm_a = np.linalg.norm(a)
    norm_b = np.linalg.norm(b)

    return dot_product / (norm_a * norm_b)

scores = []

for doc_id, doc_embedding in enumerate(document_embeddings):
    score = cosine_similarity(query_embedding, doc_embedding)
    scores.append((doc_id, score))

for doc_id, score in scores:
    print(f"Doc {doc_id}: {score:.4f} | {documents[doc_id]}")

Doc 0: 0.0224 | A dog is running in the park.
Doc 1: 0.0272 | A puppy is playing outside.
Doc 2: 0.4871 | The database stores information.
Doc 3: 0.7155 | Search engines retrieve relevant documents.
Doc 4: 0.0816 | A cat is sleeping on the sofa.


## 3. Top-K Document Ranking

After computing the cosine similarity scores between the query and all documents, we rank the documents in descending order of similarity.

The **Top-K retrieval** process consists of three steps:

1. Sort documents by their similarity scores.
2. Select the K highest-scoring documents.
3. Return the ranked documents with their similarity scores.

In this example, we retrieve the top 3 most relevant documents.

In [3]:
k = 3

ranked_results = sorted(
    scores,
    key=lambda item: item[1],
    reverse=True
)

top_k = ranked_results[:k]

In [4]:
for rank, (doc_id, score) in enumerate(top_k, start=1):
    print(f"Rank {rank}")
    print(f"Score: {score:.4f}")
    print(f"Document: {documents[doc_id]}")
    print()

Rank 1
Score: 0.7155
Document: Search engines retrieve relevant documents.

Rank 2
Score: 0.4871
Document: The database stores information.

Rank 3
Score: 0.0816
Document: A cat is sleeping on the sofa.



## 4. Building a Reusable Dense Retrieval Function

We combine the previous steps into a reusable `dense_search()` function.

Given a query and a value of K, the function:

1. Encodes the query into a dense vector.
2. Computes cosine similarity against all document embeddings.
3. Ranks documents by similarity score.
4. Returns the Top-K documents and their scores.

We then test the function using multiple queries to observe how semantic similarity affects retrieval results.

This implementation uses a Python loop to compute similarity scores, which will later be replaced by vectorized operations for more efficient retrieval.

In [5]:
def dense_search(query, k=3):
    query_embedding = model.encode(query)

    scores = []

    for doc_id, doc_embedding in enumerate(document_embeddings):
        score = cosine_similarity(query_embedding, doc_embedding)
        scores.append((doc_id, score))

    ranked_results = sorted(
        scores,
        key=lambda item: item[1],
        reverse=True
    )

    top_k = ranked_results[:k]

    return [
        (documents[doc_id], float(score))
        for doc_id, score in top_k
    ]

In [6]:
queries = [
    "How do search engines find information?",
    "A young dog playing outdoors",
    "How is data stored in a database?"
]

for query in queries:
    print(f"\nQuery: {query}")

    results = dense_search(query, k=2)

    for rank, (document, score) in enumerate(results, start=1):
        print(f"{rank}. {document}")
        print(f"   Score: {score:.4f}")


Query: How do search engines find information?
1. Search engines retrieve relevant documents.
   Score: 0.7155
2. The database stores information.
   Score: 0.4871

Query: A young dog playing outdoors
1. A puppy is playing outside.
   Score: 0.7783
2. A dog is running in the park.
   Score: 0.4189

Query: How is data stored in a database?
1. The database stores information.
   Score: 0.7581
2. Search engines retrieve relevant documents.
   Score: 0.2833


## 5. Embedding Normalization

Before implementing vectorized retrieval, we normalize all document embeddings to unit length using **L2 normalization**.

For an embedding vector $v$, L2 normalization is defined as:

$$
\hat{v} = \frac{v}{\|v\|_2}
$$

After normalization, each vector has an L2 norm of 1.

Cosine similarity is defined as:

$$
\operatorname{cosine}(q,d) =
\frac{q \cdot d}{\|q\|_2 \|d\|_2}
$$

When both embeddings are normalized:

$$
\|\hat{q}\|_2 = \|\hat{d}\|_2 = 1
$$

Therefore:

$$
\operatorname{cosine}(q,d) = \hat{q} \cdot \hat{d}
$$


In [11]:
import numpy as np

# Compute the L2 norm of each document embeddings
doc_norms = np.linalg.norm(
    document_embeddings,
    axis=1,
    keepdims=True
)

# Normalize all document embeddings
normalized_docs = document_embeddings / doc_norms

print("Original shape:", document_embeddings.shape)
print("Normalized shape:", normalized_docs.shape)
print("First document norm:", np.linalg.norm(normalized_docs[0]))

Original shape: (5, 384)
Normalized shape: (5, 384)
First document norm: 1.0


## 6. Vectorized Dense Retrieval

We implement a vectorized version of dense retrieval using NumPy matrix multiplication.

Instead of computing cosine similarity for each document individually with a Python loop, we calculate all similarity scores in a single matrix-vector operation:

$$
s = D\hat{q}
$$

where:

- $D \in \mathbb{R}^{N \times d}$ is the matrix of normalized document embeddings.
- $\hat{q} \in \mathbb{R}^{d}$ is the normalized query embedding.
- $s \in \mathbb{R}^{N}$ contains the similarity scores for all documents.

We then use `np.argsort()` to retrieve the Top-K highest-scoring documents.

### Validating the Implementation

Finally, we compare the loop-based and vectorized implementations using the same query.

We verify that both implementations return identical document rankings and numerically equivalent similarity scores using `np.isclose()`.

This confirms that vectorization preserves retrieval correctness while enabling more efficient similarity computation.

In [12]:
def dense_search_vectorized(query, k=3):
    # Encode the query
    query_embedding = model.encode(query)

    # Normalize query embedding
    query_norm = query_embedding / np.linalg.norm(query_embedding)

    # Compute all cosine similarities at once
    scores = normalized_docs @ query_norm

    # Sort scores in descending order
    top_indices = np.argsort(scores)[::-1][:k]

    # Return Top-K results
    return [
        (documents[i], float(scores[i]))
        for i in top_indices
    ]

In [13]:
query = "A young dog playing outdoors"

results_loop = dense_search(query, k=3)
results_vectorized = dense_search_vectorized(query, k=3)

print("=== Loop Version ===")
for doc, score in results_loop:
    print(f"{score:.4f} | {doc}")

print("\n=== Vectorized Version ===")
for doc, score in results_vectorized:
    print(f"{score:.4f} | {doc}")

# Check whether the results match
for (doc1, score1), (doc2, score2) in zip(
    results_loop, results_vectorized
):
    assert doc1 == doc2
    assert np.isclose(score1, score2, atol=1e-5)

print("\nBoth implementations match!")

=== Loop Version ===
0.7783 | A puppy is playing outside.
0.4189 | A dog is running in the park.
0.0181 | A cat is sleeping on the sofa.

=== Vectorized Version ===
0.7783 | A puppy is playing outside.
0.4189 | A dog is running in the park.
0.0181 | A cat is sleeping on the sofa.

Both implementations match!
